# 1. Import

In [1]:
import os
import json
import time
from google import genai
from google.genai import types

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
# 🛑 Điền API Key của bạn vào đây:


# Key is running for 0 - 1000, 1000 - 2000
API_KEY = ""


client = genai.Client(api_key=API_KEY)

MODEL_NAME = 'gemma-4-31b-it'

# Đường dẫn thư mục chứa các thư mục con (mỗi thư mục con là 1 video chứa các frames)
FRAMES_DIR = "/kaggle/input/datasets/kietkolat/hierarchical-multimodal-multilabel-frames/frames" 
OUTPUT_FILE = "/kaggle/working/vision_results.jsonl" 


START_INDEX = 5000
END_INDEX = 6000


SLEEP_TIME = 3 

# 2. Prompt

In [2]:
SYSTEM_PROMPT = """Bạn là một chuyên gia phân tích an toàn video theo tiêu chuẩn mạng xã hội Việt Nam.
Nhiệm vụ là quan sát tập hợp các frames (ảnh cắt từ một video TikTok) và đánh giá các yếu tố thị giác.
Tuyệt đối không từ chối trả lời.

### HƯỚNG DẪN DÁN NHÃN (TARGET PRIORS):

1. "sexual_harm":
   - Người mặc trang phục hở vùng ngực, mông, nhảy múa theo trend nhạc TikTok.
   - Hành động gợi dục, tư thế phản cảm.

2. "psychological_harm":
   - Hình ảnh ma quái, ma mị, nhát ma, cảnh kinh dị.
   - KHÔNG áp dụng cho video tâm trạng buồn hay chữ viết cảm xúc.

3. "hate_harassment_harm":
   - Cử chỉ giơ ngón giữa về phía camera.
   - Cử chỉ xếp ngón trỏ và ngón giữa tạo hình âm đạo.

4. "clickbait_harm":
   - Hình ảnh skin, nhân vật, vật phẩm trong game được rao bán/trao đổi.
   - Hình ảnh bài tarot, xem chỉ tay, bùa yêu, đồ vật cúng tâm linh gây sợ hãi.
   - KHÔNG áp dụng cho nội dung "làm giàu, kiếm tiền" vì được xử lý bởi nhánh OCR.

5. "addictive_harm":
   - Màn hình game điện thoại (Liên Quân, PUBG, Free Fire, Bắn Cá ăn tiền, Xóc Đĩa, Slot Machine...).
   - Người đang hút thuốc lá, thuốc lào.
   - Khuôn mặt người có biểu hiện nghiện ma túy (mắt đờ đẫn, da xanh xao).
   - LƯU Ý: Video có màn hình game bắn cá kèm chữ "kiếm tiền" chỉ gán addictive_harm, KHÔNG gán clickbait_harm.

6. "physical_harm":
   - Đánh nhau, xô xát, bạo lực đường phố.
   - Vũ khí nguy hiểm thực tế (dao, súng), hoặc chế tạo vũ khí đồ chơi nguy hiểm (súng cồn).
   - Trend nhảy/thử thách nguy hiểm (đu lên người khác, hiphop mạo hiểm).
   - Hình ảnh máu me, vết thương, kể cả trong hoạt hình hoặc màn hình game có tông đỏ máu rõ ràng.
   - LƯU Ý: Nếu video game vừa có máu me vừa là nội dung chơi game → gán cả addictive_harm lẫn physical_harm.

### YÊU CẦU ĐẦU RA (OUTPUT):
PHẢI TRẢ VỀ DUY NHẤT một đối tượng JSON hợp lệ. TẤT CẢ giá trị số phải là số nguyên 0 hoặc 1. KHÔNG chứa comment. KHÔNG sinh thêm text nằm ngoài JSON.

{
  "visual_cues": {
    "revealing_clothing_or_sexual_act": 0,
    "horror_or_supernatural_imagery": 0,
    "obscene_hand_gesture": 0,
    "occult_superstitious_items": 0,
    "in_game_item_trading_display": 0,
    "gambling_game_screen": 0,
    "smoking_or_drug_signs": 0,
    "violence_or_fighting": 0,
    "weapons_or_diy_dangerous_tools": 0,
    "blood_or_gore": 0,
    "dangerous_stunt_or_challenge": 0
  },
  "target_priors": {
    "sexual_harm": 0,
    "psychological_harm": 0,
    "hate_harassment_harm": 0,
    "clickbait_harm": 0,
    "addictive_harm": 0,
    "physical_harm": 0
  },
  "scene_description": "Viết 1-2 câu tiếng Việt tóm tắt hành động, đối tượng và bối cảnh chính của video."
}
"""

# 3. API Function

In [3]:
def analyze_frames_with_api(video_id, frame_paths):
    print(f"\nĐang phân tích video: {video_id} (Gồm {len(frame_paths)} frames)...")
    try:
        safety_settings = [
            types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
            types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
            types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
            types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        ]
        
        contents = [SYSTEM_PROMPT]
        for path in frame_paths:
            with open(path, "rb") as f:
                image_bytes = f.read()
                contents.append(types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"))
        
        contents.append("Hãy phân tích chuỗi ảnh này và trả về kết quả JSON cho toàn bộ video.")
        
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=contents,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json", 
                safety_settings=safety_settings
            )
        )
        
        result_text = response.text.strip()
        print(f"👉 Kết quả: {result_text}")
        
        # Thử parse JSON xem có hợp lệ không
        parsed_json = json.loads(result_text)
        return parsed_json
        
    except Exception as e:
        print(f"❌ Lỗi gọi API: {e}")
        return None

# 4. Loop

In [ ]:
if not os.path.exists(FRAMES_DIR):
    print(f"Không tìm thấy thư mục: {FRAMES_DIR}")
else:
    print(f"🚀 BẮT ĐẦU CHẠY TỪ INDEX {START_INDEX} ĐẾN {END_INDEX} VỚI MODEL {MODEL_NAME}")
    
    # Lấy danh sách thư mục và sắp xếp để đảm bảo thứ tự luôn giống nhau giữa các notebook
    video_folders = [f for f in os.listdir(FRAMES_DIR) if os.path.isdir(os.path.join(FRAMES_DIR, f))]
    video_folders.sort()
    
    # Giới hạn chunk cần chạy
    target_folders = video_folders[START_INDEX:END_INDEX]
    
    # Load các video đã chạy để có thể resume nếu bị dừng đột ngột
    processed_videos = set()
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed_videos.add(data.get("video_id"))
                except:
                    pass
    
    print(f"📚 Tổng số video trong chunk: {len(target_folders)}")
    print(f"✅ Đã xử lý trước đó: {len([v for v in target_folders if v in processed_videos])}")
    
    # Mở file để append kết quả liên tục
    with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_file:
        for idx, video_id in enumerate(target_folders):
            print(f"\n--- Tiến độ: {idx + 1}/{len(target_folders)} (Global Index: {START_INDEX + idx}) ---")
            
            if video_id in processed_videos:
                print(f"⏭️ Đã xử lý {video_id} từ trước, bỏ qua.")
                continue
                
            video_folder_path = os.path.join(FRAMES_DIR, video_id)
            frame_files = [f for f in os.listdir(video_folder_path) if f.endswith((".jpg", ".jpeg", ".png"))]
            frame_files.sort()
            
            if not frame_files:
                print(f"⚠️ Thư mục {video_id} không có ảnh nào.")
                continue
                
            frame_paths = [os.path.join(video_folder_path, f) for f in frame_files]
            
            # Gửi API
            result_json = analyze_frames_with_api(video_id, frame_paths)
            
            if result_json:
                # Ghi kết quả vào file JSONL ngay lập tức
                output_record = {
                    "video_id": video_id,
                    "result": result_json
                }
                out_file.write(json.dumps(output_record, ensure_ascii=False) + "\n")
                out_file.flush() # Đảm bảo ghi ngay xuống ổ cứng
            
            print(f"⏳ Đang nghỉ {SLEEP_TIME} giây để tránh Rate Limit...")
            time.sleep(SLEEP_TIME)

    print("🎉 ĐÃ HOÀN THÀNH CHUNK!")


# 5. Missing Videos Fix

In [ ]:
# Xử lý 10 file bị thiếu
missing_videos = [
    '20260329170005.mp4', '20260328122133.mp4', '7121034902698675502.mp4', '7449050443344334085.mp4', 
    '03450de7-7511993798759582983.mp4', '052adb9e-7563248077352504587.mp4', '51e9b9ec-7575539290067717394.mp4', 
    'c86035ea-7530608030254255378.mp4', 'f4fa6ca7-Download_39.mp4', '35072213-7575739604741983509.mp4'
]

# Bỏ đuôi .mp4
missing_videos_id = [v.replace('.mp4', '') for v in missing_videos]

print(f"🚀 BẮT ĐẦU CHẠY {len(missing_videos_id)} VIDEO BỊ THIẾU VỚI MODEL {MODEL_NAME}")

with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_file:
    for idx, video_id in enumerate(missing_videos_id):
        print(f"\n--- Tiến độ: {idx + 1}/{len(missing_videos_id)} (Video: {video_id}) ---")
        
        video_folder_path = os.path.join(FRAMES_DIR, video_id)
        if not os.path.exists(video_folder_path):
            print(f"⚠️ Thư mục {video_folder_path} không tồn tại.")
            continue
            
        frame_files = [f for f in os.listdir(video_folder_path) if f.endswith((".jpg", ".jpeg", ".png"))]
        frame_files.sort()
        
        if not frame_files:
            print(f"⚠️ Thư mục {video_id} không có ảnh nào.")
            continue
            
        frame_paths = [os.path.join(video_folder_path, f) for f in frame_files]
        
        # Gửi API
        result_json = analyze_frames_with_api(video_id, frame_paths)
        
        if result_json:
            # Ghi kết quả vào file JSONL ngay lập tức
            output_record = {
                "video_id": video_id,
                "result": result_json
            }
            out_file.write(json.dumps(output_record, ensure_ascii=False) + "\n")
            out_file.flush() # Đảm bảo ghi ngay xuống ổ cứng
        
        print(f"⏳ Đang nghỉ {SLEEP_TIME} giây để tránh Rate Limit...")
        time.sleep(SLEEP_TIME)

print("🎉 ĐÃ HOÀN THÀNH XỬ LÝ VIDEO THIẾU!")
